In [0]:
%run ./99_Config

In [0]:
%run ./98_Utilities

In [0]:
from pyspark.sql import Row
from pyspark.sql.functions import *
from datetime import datetime
import traceback

In [0]:
def initialize_logging():

    dbutils.fs.mkdirs(LOG_PATH)

    dbutils.fs.mkdirs(f"{LOG_PATH}/pipeline_logs")

    dbutils.fs.mkdirs(f"{LOG_PATH}/error_logs")

    dbutils.fs.mkdirs(f"{LOG_PATH}/audit_logs")

    print("Logging folders initialized.")

In [0]:
def log_success(notebook,
                dataset,
                rows_read,
                rows_written,
                execution_time):

    print("="*80)

    print("STATUS           : SUCCESS")

    print(f"Notebook         : {notebook}")

    print(f"Dataset          : {dataset}")

    print(f"Rows Read        : {rows_read}")

    print(f"Rows Written     : {rows_written}")

    print(f"Execution Time   : {execution_time}")

    print("="*80)

In [0]:
def log_warning(message):

    print("="*80)

    print("WARNING")

    print(message)

    print("="*80)

In [0]:
def log_error(notebook,
              dataset,
              exception):

    print("="*80)

    print("STATUS : FAILED")

    print(f"Notebook : {notebook}")

    print(f"Dataset  : {dataset}")

    print(str(exception))

    print("="*80)

In [0]:
def print_stacktrace():

    print(traceback.format_exc())

In [0]:
def create_audit_dataframe(notebook,
                           dataset,
                           rows_read,
                           rows_written,
                           rejected_rows,
                           status,
                           error_message=""):

    row = Row(

        pipeline_name = PIPELINE_NAME,

        pipeline_run_id = CURRENT_RUN_ID,

        notebook_name = notebook,

        dataset = dataset,

        rows_read = rows_read,

        rows_written = rows_written,

        rejected_rows = rejected_rows,

        status = status,

        error_message = error_message,

        execution_time = str(datetime.now())

    )

    return spark.createDataFrame([row])

In [0]:
def save_audit_log(df):

    audit_path = f"{LOG_PATH}/audit_logs"

    (

        df.write

        .format("delta")

        .mode("append")

        .save(audit_path)

    )

    create_delta_table(

        "audit_pipeline_log",

        audit_path

    )

In [0]:
def pipeline_start(notebook):

    print("="*80)

    print(f"Starting {notebook}")

    print(f"Pipeline : {PIPELINE_NAME}")

    print(f"Run ID   : {CURRENT_RUN_ID}")

    print(f"Start    : {datetime.now()}")

    print("="*80)

In [0]:
def pipeline_end(notebook):

    print("="*80)

    print(f"Completed {notebook}")

    print(f"End Time : {datetime.now()}")

    print("="*80)

In [0]:
def audit(notebook,
          dataset,
          rows_read,
          rows_written,
          rejected_rows,
          status,
          error=""):

    audit_df = create_audit_dataframe(

        notebook,

        dataset,

        rows_read,

        rows_written,

        rejected_rows,

        status,

        error

    )

    save_audit_log(audit_df)

In [0]:
initialize_logging()

pipeline_start("97_Logger")

log_success(

    notebook="97_Logger",

    dataset="Demo",

    rows_read=100,

    rows_written=100,

    execution_time="2 sec"

)

audit(

    notebook="97_Logger",

    dataset="Demo",

    rows_read=100,

    rows_written=100,

    rejected_rows=0,

    status="SUCCESS"

)

pipeline_end("97_Logger")